# **Trial Activation Analysis SQL Production Models**
## **Splendor Analytics Data Challenge**

**Author:** Ehijele David  
**Notebook:** 02 SQL Production Models  
**Challenge:** Splendor Analytics Data Challenge  
**Date:** 23/03/2026

---

This notebook builds and validates the SQL production models that operationalize the trial activation framework which has been defined in Notebook 1. The system includes three models which operate through two layers of a data warehouse architecture:

**Staging Layer**
- `stg_events.sql` — cleans and enriches the raw event data, creating
  all derived fields that downstream models depend on

**Marts Layer**
- `trial_goals.sql` — tracks whether each organisation completed each
  of the four trial goals defined in the analysis
- `trial_activation.sql` — tracks which organisations achieved full
  trial activation and whether they converted to paying customers

Each model exists as an independent SQL file which developers have stored in the
repository's `sql/` directory. The repository maintains dbt-style conventions to
structure the codebase and establish naming conventions and documentation standards.
Researchers use DuckDB to run their tests which examine every model against the
cleaned dataset to verify that both output results and final models achieve
production-level standards.

The trial goal definitions, activity conditions, and thresholds used
in these models are derived from the Python analysis in
`01_data_cleaning_and_analysis.ipynb`.

## **Section 1 Environment Setup**

DuckDB enables SQL model execution on CSV data files without needing an active database. The SQL code requires standard warehouse-compatible syntax for its implementation because it will be tested in a Colab environment.

In [34]:
# ── Section 1: Environment Setup ─────────────────────────────────────────────

# Install DuckDB
!pip install duckdb --quiet

import duckdb

# ── Step 1: Clone or update the GitHub repository ────────────────────────────
import os
import subprocess

REPO_URL  = 'https://github.com/EEHZYDHAVE/splendor-trial-activation-model.git'
REPO_NAME = 'splendor-trial-activation-model'

if os.path.exists(REPO_NAME):
    print(f"Repository already exists — pulling latest changes...")
    result = subprocess.run(
        ['git', '-C', REPO_NAME, 'pull'],
        capture_output=True, text=True
    )
else:
    print(f"Cloning repository...")
    result = subprocess.run(
        ['git', 'clone', REPO_URL],
        capture_output=True, text=True
    )

print(result.stdout if result.stdout else result.stderr)
print("Repository ready.")

# ── Step 2: Mount Google Drive and locate raw data ───────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ============================================================
# CONFIGURATION — change only this path if data moves
# ============================================================
RAW_DATA_PATH = '/content/drive/MyDrive/Advanced Data Analytics/Splendor Data Challenge/DA task.csv'

# ── Step 3: Load raw data directly into DuckDB ───────────────────────────────
con = duckdb.connect()

con.execute(f"""
    CREATE OR REPLACE TABLE raw_events AS
    SELECT *
    FROM read_csv_auto(
        '{RAW_DATA_PATH}',
        header     = true,
        dateformat = '%Y-%m-%d %H:%M:%S'
    )
""")

# ── Step 4: Normalise column names to lowercase ───────────────────────────────
# Handles uppercase, lowercase, or mixed case source data automatically
existing_cols = con.execute(
    "DESCRIBE raw_events"
).df()['column_name'].tolist()

for col in existing_cols:
    if col != col.lower():
        con.execute(
            f'ALTER TABLE raw_events RENAME COLUMN "{col}" TO "{col.lower()}"'
        )

print("Column names normalised to lowercase.")

# ── Step 5: Verify load ───────────────────────────────────────────────────────
print("\n=== ENVIRONMENT SETUP COMPLETE ===")
print(f"DuckDB version: {duckdb.__version__}")

result = con.execute("""
    SELECT
        COUNT(*)                        AS total_rows,
        COUNT(DISTINCT organization_id) AS unique_orgs,
        COUNT(DISTINCT activity_name)   AS unique_activities,
        MIN(timestamp)                  AS earliest_event,
        MAX(timestamp)                  AS latest_event
    FROM raw_events
""").df()

print(f"\nRaw data loaded into DuckDB as table: raw_events")
print(f"Total rows:         {result['total_rows'][0]:,}")
print(f"Unique orgs:        {result['unique_orgs'][0]:,}")
print(f"Unique activities:  {result['unique_activities'][0]:,}")
print(f"Date range:         "
      f"{result['earliest_event'][0]} → {result['latest_event'][0]}")

print("\nFinal column names:")
print(con.execute("DESCRIBE raw_events").df()[
    ['column_name', 'column_type']
].to_string())

Repository already exists — pulling latest changes...
Updating 67aa141..1ffa18f
Fast-forward
 sql/marts/trial_activation.sql | 122 +++++++++++++++++++++++++++++++++++++++++
 1 file changed, 122 insertions(+)

Repository ready.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Column names normalised to lowercase.

=== ENVIRONMENT SETUP COMPLETE ===
DuckDB version: 1.3.2

Raw data loaded into DuckDB as table: raw_events
Total rows:         170,526
Unique orgs:        966
Unique activities:  28
Date range:         2024-01-01 20:52:26 → 2024-04-28 15:10:31

Final column names:
       column_name column_type
0  organization_id     VARCHAR
1    activity_name     VARCHAR
2        timestamp   TIMESTAMP
3        converted     BOOLEAN
4     converted_at   TIMESTAMP
5      trial_start   TIMESTAMP
6        trial_end   TIMESTAMP


## **Section 2 Staging Layer (`stg_events`)**

The staging layer establishes the basic structure which supports all components of the model. The system processes complete raw event data through its cleaning and type casting and enrichment procedures to create a single output that downstream mart models can use without accessing raw data. The system will automatically clean all new raw data that enters this layer without requiring human operators to perform any tasks.

This model handles: timestamp casting, trial window filtering,
day_of_trial calculation, activity bucket assignment, conversion
timing classification, and deduplication of known instrumentation
events. Any raw data flowing through this layer is automatically
cleaned without manual intervention.

The SQL lives in `sql/staging/stg_events.sql` in the repository.

In [35]:
# ── Section 2: Staging Layer — stg_events ────────────────────────────────────

# Read SQL file from cloned repository
with open(f'{REPO_NAME}/sql/staging/stg_events.sql', 'r') as f:
    stg_events_sql = f.read()

# Create staging view in DuckDB
con.execute(f"CREATE OR REPLACE VIEW stg_events AS {stg_events_sql}")

# ── Validation ────────────────────────────────────────────────────────────────
print("=== STG_EVENTS VALIDATION ===")

print(con.execute("""
    SELECT
        COUNT(*)                        AS total_rows,
        COUNT(DISTINCT organization_id) AS unique_orgs,
        COUNT(DISTINCT activity_name)   AS unique_activities
    FROM stg_events
""").df().to_string())

print("\n=== DAY OF TRIAL RANGE ===")
print(con.execute("""
    SELECT
        MIN(day_of_trial)                                    AS min_day,
        MAX(day_of_trial)                                    AS max_day,
        COUNT(*) FILTER (WHERE day_of_trial < 0)             AS pre_trial_events,
        COUNT(*) FILTER (WHERE day_of_trial > 30)            AS post_trial_events,
        COUNT(*) FILTER (WHERE day_of_trial BETWEEN 0 AND 30) AS valid_events
    FROM stg_events
""").df().to_string())

print("\n=== CONVERSION TIMING DISTRIBUTION ===")
print(con.execute("""
    SELECT
        conversion_timing,
        COUNT(DISTINCT organization_id) AS organisations
    FROM stg_events
    GROUP BY conversion_timing
    ORDER BY organisations DESC
""").df().to_string())

print("\n=== BUCKET DISTRIBUTION ===")
print(con.execute("""
    SELECT
        activity_bucket,
        COUNT(*)                                                    AS events,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)         AS pct_of_events
    FROM stg_events
    GROUP BY activity_bucket
    ORDER BY activity_bucket
""").df().to_string())

print("\n=== NULL CHECK ===")
print(con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE organization_id  IS NULL) AS null_org_id,
        COUNT(*) FILTER (WHERE activity_name    IS NULL) AS null_activity,
        COUNT(*) FILTER (WHERE event_timestamp  IS NULL) AS null_timestamp,
        COUNT(*) FILTER (WHERE activity_bucket  IS NULL) AS null_bucket
    FROM stg_events
""").df().to_string())

print("\n=== SAMPLE OUTPUT ===")
print(con.execute("""
    SELECT
        organization_id,
        activity_name,
        activity_bucket,
        day_of_trial,
        converted,
        conversion_timing,
        trial_cohort_month
    FROM stg_events
    LIMIT 5
""").df().to_string())

=== STG_EVENTS VALIDATION ===
   total_rows  unique_orgs  unique_activities
0      169335          966                 28

=== DAY OF TRIAL RANGE ===
   min_day  max_day  pre_trial_events  post_trial_events  valid_events
0        0       30                 0                  0        169335

=== CONVERSION TIMING DISTRIBUTION ===
  conversion_timing  organisations
0     not_converted            760
1      within_trial            105
2  post_trial_short             63
3   post_trial_long             38

=== BUCKET DISTRIBUTION ===
   activity_bucket  events  pct_of_events
0                1  149423           88.2
1                2   15676            9.3
2                3    4236            2.5

=== NULL CHECK ===
   null_org_id  null_activity  null_timestamp  null_bucket
0            0              0               0            0

=== SAMPLE OUTPUT ===
                    organization_id           activity_name  activity_bucket  day_of_trial  converted conversion_timing trial_cohort_mo

**Staging Layer Validated**

The `stg_events` model produces 169335 clean events which it processes across 966 organizations and 28 activity types. The events exist within the valid testing period which starts from Day 0 and ends on Day 30. All essential columns maintain 100 percent data integrity because there are no null values present. The system properly categorizes all 28 activities into their designated activity buckets.

The system converts 760 cases which remain unprocessed while 105 cases operate within their testing period and 63 cases exist after their testing period ends and 38 cases operate after their testing period ends. The post-trial short and long buckets have a small 2-organization difference which occurs because pandas and DuckDB handle sub-day timestamp precision differently according to the documentation in the SQL file.

The staging layer is production-ready. All downstream mart models
read exclusively from `stg_events`.

## **Section 3 Trial Goals Mart (`trial_goals`)**

The trial goals mart tracks whether each organisation completed
each of the four trial goals defined in the analysis. The table
displays one entry for every organisation which needs to achieve
four goals because the organisation requires four goals to complete. The grain is intentionally
long-format so the table can accommodate any number of goals
without a schema change.

The SQL lives in `sql/marts/trial_goals.sql` in the repository.

In [36]:
# ── Section 3: Trial Goals Mart — trial_goals ─────────────────────────────────

import re

# Read SQL file from cloned repository
with open(f'{REPO_NAME}/sql/marts/trial_goals.sql', 'r') as f:
    trial_goals_sql = f.read()

# Strip single-line comments before passing to DuckDB
# Comments are preserved in the .sql file for documentation
# but DuckDB view creation does not support them inline
trial_goals_sql_clean = re.sub(r'--[^\n]*', '', trial_goals_sql)
trial_goals_sql_clean = re.sub(r'\s+', ' ', trial_goals_sql_clean).strip()

# Create as a view in DuckDB
con.execute(f"CREATE OR REPLACE VIEW trial_goals AS {trial_goals_sql_clean}")

print("trial_goals view created successfully.")

# ── Validation ────────────────────────────────────────────────────────────────
print("=== TRIAL_GOALS VALIDATION ===")

print(con.execute("""
    SELECT
        COUNT(*)                        AS total_rows,
        COUNT(DISTINCT organization_id) AS unique_orgs,
        COUNT(DISTINCT goal_name)       AS unique_goals
    FROM trial_goals
""").df().to_string())

print("\n=== GOAL COMPLETION RATES ===")
print(con.execute("""
    SELECT
        goal_name,
        COUNT(*)                                        AS total_orgs,
        SUM(CASE WHEN is_completed THEN 1 ELSE 0 END)  AS completed,
        ROUND(
            SUM(CASE WHEN is_completed THEN 1 ELSE 0 END) * 100.0
            / COUNT(*), 1
        )                                               AS completion_rate_pct,
        ROUND(
            AVG(CASE WHEN is_completed
                THEN CAST(converted AS INTEGER) END) * 100, 1
        )                                               AS conversion_rate_pct
    FROM trial_goals
    GROUP BY goal_name
    ORDER BY goal_name
""").df().to_string())

print("\n=== GRAIN CHECK — rows per organisation ===")
print(con.execute("""
    SELECT
        rows_per_org,
        COUNT(*) AS org_count
    FROM (
        SELECT
            organization_id,
            COUNT(*) AS rows_per_org
        FROM trial_goals
        GROUP BY organization_id
    )
    GROUP BY rows_per_org
    ORDER BY rows_per_org
""").df().to_string())

print("\n=== NULL CHECK ===")
print(con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE organization_id IS NULL) AS null_org_id,
        COUNT(*) FILTER (WHERE goal_name       IS NULL) AS null_goal_name,
        COUNT(*) FILTER (WHERE is_completed    IS NULL) AS null_is_completed
    FROM trial_goals
""").df().to_string())

print("\n=== SAMPLE OUTPUT ===")
print(con.execute("""
    SELECT
        organization_id,
        goal_name,
        is_completed,
        day_goal_completed,
        converted,
        conversion_timing
    FROM trial_goals
    LIMIT 8
""").df().to_string())

trial_goals view created successfully.
=== TRIAL_GOALS VALIDATION ===
   total_rows  unique_orgs  unique_goals
0        3864          966             4

=== GOAL COMPLETION RATES ===
                     goal_name  total_orgs  completed  completion_rate_pct  conversion_rate_pct
0    goal_1_schedule_published         966      848.0                 87.8                 21.8
1      goal_2_team_operational         966      211.0                 21.8                 22.7
2   goal_3_management_decision         966      214.0                 22.2                 22.0
3  goal_4_sustained_engagement         966      253.0                 26.2                 23.3

=== GRAIN CHECK — rows per organisation ===
   rows_per_org  org_count
0             4        966

=== NULL CHECK ===
   null_org_id  null_goal_name  null_is_completed
0            0               0                  0

=== SAMPLE OUTPUT ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                    organization_id                    goal_name  is_completed  day_goal_completed  converted conversion_timing
0  0040dd9ab132b92d5d04bc3acf14d2e2    goal_1_schedule_published          True                   0      False     not_converted
1  0040dd9ab132b92d5d04bc3acf14d2e2      goal_2_team_operational          True                  26      False     not_converted
2  0040dd9ab132b92d5d04bc3acf14d2e2   goal_3_management_decision          True                  23      False     not_converted
3  0040dd9ab132b92d5d04bc3acf14d2e2  goal_4_sustained_engagement          True                   0      False     not_converted
4  00456fd86311b6095ad05f7e31758f0d    goal_1_schedule_published          True                   0      False     not_converted
5  00456fd86311b6095ad05f7e31758f0d      goal_2_team_operational         False                <NA>      False     not_converted
6  00456fd86311b6095ad05f7e31758f0d   goal_3_management_decision         False                <NA>      

**trial_goals Validated**

The model produces 3,864 rows, exactly 4 rows per organisation across 966 organisations. All goal completion rates and conversion rates match the Python analysis in Notebook 1 exactly. The data contains no missing values for essential columns. The long-format grain is confirmed because every organisation appears exactly four times in the dataset which includes all completion status results.

The system has reached production readiness and is available for use by the `trial_activation` mart.

## **Section 4 Trial Activation Mart (`trial_activation`)**

The trial activation mart is the business-facing summary table.
It produces one row per organisation and answers the most important
question directly: has this organisation achieved trial activation,
and did they convert? This is the table a dashboard or executive
report would query.

The SQL lives in `sql/marts/trial_activation.sql` in the repository.

In [37]:
# ── Section 4: Trial Activation Mart — trial_activation ──────────────────────

import re

# Read SQL file from cloned repository
with open(f'{REPO_NAME}/sql/marts/trial_activation.sql', 'r') as f:
    trial_activation_sql = f.read()

# Strip comments for DuckDB execution
trial_activation_sql_clean = re.sub(r'--[^\n]*', '', trial_activation_sql)
trial_activation_sql_clean = re.sub(r'\s+', ' ', trial_activation_sql_clean).strip()

# Create as a view in DuckDB
con.execute(
    f"CREATE OR REPLACE VIEW trial_activation AS {trial_activation_sql_clean}"
)
print("trial_activation view created successfully.")

# ── Validation ────────────────────────────────────────────────────────────────
print("\n=== TRIAL_ACTIVATION VALIDATION ===")

print(con.execute("""
    SELECT
        COUNT(*)                        AS total_rows,
        COUNT(DISTINCT organization_id) AS unique_orgs,
        SUM(CASE WHEN is_activated  THEN 1 ELSE 0 END) AS activated,
        SUM(CASE WHEN converted     THEN 1 ELSE 0 END) AS converted
    FROM trial_activation
""").df().to_string())

print("\n=== GRAIN CHECK — one row per organisation ===")
print(con.execute("""
    SELECT
        rows_per_org,
        COUNT(*) AS org_count
    FROM (
        SELECT organization_id, COUNT(*) AS rows_per_org
        FROM trial_activation
        GROUP BY organization_id
    )
    GROUP BY rows_per_org
""").df().to_string())

print("\n=== ACTIVATION AND CONVERSION RATES ===")
print(con.execute("""
    SELECT
        is_activated,
        COUNT(*)                                            AS organisations,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct_of_total,
        SUM(CASE WHEN converted THEN 1 ELSE 0 END)         AS converted,
        ROUND(
            SUM(CASE WHEN converted THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*), 1
        )                                                   AS conversion_rate_pct
    FROM trial_activation
    GROUP BY is_activated
    ORDER BY is_activated
""").df().to_string())

print("\n=== ACTIVATION BY COHORT MONTH ===")
print(con.execute("""
    SELECT
        trial_cohort_month,
        COUNT(*)                                                AS total_orgs,
        SUM(CASE WHEN is_activated THEN 1 ELSE 0 END)          AS activated,
        ROUND(
            SUM(CASE WHEN is_activated THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*), 1
        )                                                       AS activation_rate_pct,
        ROUND(
            SUM(CASE WHEN converted THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*), 1
        )                                                       AS conversion_rate_pct
    FROM trial_activation
    GROUP BY trial_cohort_month
    ORDER BY trial_cohort_month
""").df().to_string())

print("\n=== NULL CHECK ===")
print(con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE organization_id IS NULL) AS null_org_id,
        COUNT(*) FILTER (WHERE is_activated    IS NULL) AS null_is_activated,
        COUNT(*) FILTER (WHERE converted       IS NULL) AS null_converted
    FROM trial_activation
""").df().to_string())

print("\n=== SAMPLE OUTPUT ===")
print(con.execute("""
    SELECT
        organization_id,
        trial_cohort_month,
        is_activated,
        day_activated,
        goals_completed,
        converted,
        conversion_timing
    FROM trial_activation
    LIMIT 8
""").df().to_string())

trial_activation view created successfully.

=== TRIAL_ACTIVATION VALIDATION ===
   total_rows  unique_orgs  activated  converted
0         966          966      104.0      206.0

=== GRAIN CHECK — one row per organisation ===
   rows_per_org  org_count
0             1        966

=== ACTIVATION AND CONVERSION RATES ===
   is_activated  organisations  pct_of_total  converted  conversion_rate_pct
0         False            862          89.2      184.0                 21.3
1          True            104          10.8       22.0                 21.2

=== ACTIVATION BY COHORT MONTH ===
  trial_cohort_month  total_orgs  activated  activation_rate_pct  conversion_rate_pct
0         2024-01-01         305       39.0                 12.8                 23.0
1         2024-02-01         347       32.0                  9.2                 22.8
2         2024-03-01         314       33.0                 10.5                 18.2

=== NULL CHECK ===
   null_org_id  null_is_activated  null_convert

**trial_activation Validated**

The model generates 966 rows which represent one complete set for every organization. The 104 organizations show operational status at 10.8% while their conversion rate matches the Python analysis results from Notebook 1 with 21.2% conversion. The three monthly cohorts show identical patterns for both activation and conversion rates at the cohort level. The data set contains no missing values for any essential field.

The table serves as the main business output for the complete pipeline operation. A single query against this table answers the core business question: which organisations have achieved trial activation and did they convert?

## **Section 5 Model Validation Summary**

The last cross-model reconciliation process shows that all three SQL models maintain internal consistency with each other and the Python analysis conducted in Notebook 1. The models achieve production-ready status for submission only when all validation tests successfully pass.

In [38]:
# ── Section 5: Model Validation Summary ──────────────────────────────────────

print("=" * 65)
print("CROSS-MODEL RECONCILIATION")
print("=" * 65)

# ── Check 1: Row counts consistent across models ──────────────────────────────
print("\n--- CHECK 1: ROW COUNTS ---")
stg_rows  = con.execute(
    "SELECT COUNT(*) AS n FROM stg_events"
).df()['n'][0]
tg_orgs   = con.execute(
    "SELECT COUNT(DISTINCT organization_id) AS n FROM trial_goals"
).df()['n'][0]
ta_rows   = con.execute(
    "SELECT COUNT(*) AS n FROM trial_activation"
).df()['n'][0]
tg_rows   = con.execute(
    "SELECT COUNT(*) AS n FROM trial_goals"
).df()['n'][0]

print(f"  stg_events total rows:              {stg_rows:,}")
print(f"  trial_goals total rows:             {tg_rows:,}  "
      f"(expected {tg_orgs * 4:,} = {tg_orgs} orgs × 4 goals)")
print(f"  trial_activation total rows:        {ta_rows:,}  "
      f"(expected 966)")
print(f"  trial_goals grain check:            "
      f"{'PASS ✓' if tg_rows == tg_orgs * 4 else 'FAIL ✗'}")
print(f"  trial_activation grain check:       "
      f"{'PASS ✓' if ta_rows == 966 else 'FAIL ✗'}")

# ── Check 2: Organisation counts consistent ───────────────────────────────────
print("\n--- CHECK 2: ORGANISATION COUNTS ---")
stg_orgs = con.execute(
    "SELECT COUNT(DISTINCT organization_id) AS n FROM stg_events"
).df()['n'][0]
ta_orgs  = con.execute(
    "SELECT COUNT(DISTINCT organization_id) AS n FROM trial_activation"
).df()['n'][0]

print(f"  stg_events unique orgs:             {stg_orgs:,}")
print(f"  trial_goals unique orgs:            {tg_orgs:,}")
print(f"  trial_activation unique orgs:       {ta_orgs:,}")
print(f"  All models consistent:              "
      f"{'PASS ✓' if stg_orgs == tg_orgs == ta_orgs else 'FAIL ✗'}")

# ── Check 3: Conversion counts consistent ────────────────────────────────────
print("\n--- CHECK 3: CONVERSION COUNTS ---")
stg_conv = con.execute("""
    SELECT COUNT(DISTINCT organization_id) AS n
    FROM stg_events WHERE converted = TRUE
""").df()['n'][0]
ta_conv  = con.execute("""
    SELECT COUNT(*) AS n
    FROM trial_activation WHERE converted = TRUE
""").df()['n'][0]

print(f"  stg_events converted orgs:          {stg_conv:,}")
print(f"  trial_activation converted orgs:    {ta_conv:,}")
print(f"  Consistent:                         "
      f"{'PASS ✓' if stg_conv == ta_conv else 'FAIL ✗'}")

# ── Check 4: Activation count matches goal completion ─────────────────────────
print("\n--- CHECK 4: ACTIVATION CONSISTENCY ---")
ta_activated = con.execute("""
    SELECT COUNT(*) AS n
    FROM trial_activation WHERE is_activated = TRUE
""").df()['n'][0]
tg_all_complete = con.execute("""
    SELECT COUNT(*) AS n FROM (
        SELECT organization_id
        FROM trial_goals
        GROUP BY organization_id
        HAVING SUM(CASE WHEN is_completed THEN 1 ELSE 0 END) = 4
    )
""").df()['n'][0]

print(f"  trial_activation activated orgs:    {ta_activated:,}")
print(f"  trial_goals all-4-complete orgs:    {tg_all_complete:,}")
print(f"  Consistent:                         "
      f"{'PASS ✓' if ta_activated == tg_all_complete else 'FAIL ✗'}")

# ── Check 5: Notebook 1 reconciliation ───────────────────────────────────────
print("\n--- CHECK 5: NOTEBOOK 1 RECONCILIATION ---")
notebook1_targets = {
    'Total organisations':      966,
    'Converted organisations':  206,
    'Activated organisations':  104,
    'Activation rate (%)':      10.8,
    'Activated conversion (%)': 21.2,
}

ta_summary = con.execute("""
    SELECT
        COUNT(*)                                                AS total_orgs,
        SUM(CASE WHEN converted    THEN 1 ELSE 0 END)          AS converted_orgs,
        SUM(CASE WHEN is_activated THEN 1 ELSE 0 END)          AS activated_orgs,
        ROUND(
            SUM(CASE WHEN is_activated THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*), 1
        )                                                       AS activation_rate,
        ROUND(
            SUM(CASE WHEN is_activated AND converted
                THEN 1 ELSE 0 END)
            * 100.0
            / NULLIF(SUM(CASE WHEN is_activated
                THEN 1 ELSE 0 END), 0), 1
        )                                                       AS activated_conv_rate
    FROM trial_activation
""").df()

sql_values = {
    'Total organisations':      ta_summary['total_orgs'][0],
    'Converted organisations':  ta_summary['converted_orgs'][0],
    'Activated organisations':  ta_summary['activated_orgs'][0],
    'Activation rate (%)':      ta_summary['activation_rate'][0],
    'Activated conversion (%)': ta_summary['activated_conv_rate'][0],
}

print(f"  {'Metric':<35} {'Notebook 1':>12} {'SQL Model':>12} {'Status':>8}")
print(f"  {'-'*67}")
all_pass = True
for metric, nb1_val in notebook1_targets.items():
    sql_val = sql_values[metric]
    status  = 'PASS ✓' if sql_val == nb1_val else 'FAIL ✗'
    if status == 'FAIL ✗':
        all_pass = False
    print(f"  {metric:<35} {nb1_val:>12} {sql_val:>12} {status:>8}")

print(f"\n{'=' * 65}")
print(f"OVERALL RESULT: {'ALL CHECKS PASSED ✓' if all_pass else 'SOME CHECKS FAILED ✗'}")
print(f"{'=' * 65}")

CROSS-MODEL RECONCILIATION

--- CHECK 1: ROW COUNTS ---
  stg_events total rows:              169,335
  trial_goals total rows:             3,864  (expected 3,864 = 966 orgs × 4 goals)
  trial_activation total rows:        966  (expected 966)
  trial_goals grain check:            PASS ✓
  trial_activation grain check:       PASS ✓

--- CHECK 2: ORGANISATION COUNTS ---
  stg_events unique orgs:             966
  trial_goals unique orgs:            966
  trial_activation unique orgs:       966
  All models consistent:              PASS ✓

--- CHECK 3: CONVERSION COUNTS ---
  stg_events converted orgs:          206
  trial_activation converted orgs:    206
  Consistent:                         PASS ✓

--- CHECK 4: ACTIVATION CONSISTENCY ---
  trial_activation activated orgs:    104
  trial_goals all-4-complete orgs:    104
  Consistent:                         PASS ✓

--- CHECK 5: NOTEBOOK 1 RECONCILIATION ---
  Metric                                Notebook 1    SQL Model   Status
  ----

**All checks passed. The SQL production models are validated and
production-ready.**

The SQL production models have received validation which confirms their readiness for production use according to their established testing criteria.

The three models demonstrate internal consistency by matching both their own results and the Python analysis results from Notebook 1. The three data sets stg_events, trial_goals, and trial_activation show complete agreement on row counts, organisation counts, conversion counts, activation counts, and all key metrics.

The models are ready for submission in the GitHub repository.

In [42]:
import nbformat

notebook_path = '/content/drive/MyDrive/Advanced Data Analytics/Splendor Data Challenge/sql_production_models.ipynb'  # update path

with open(notebook_path, 'r', encoding='utf-8') as f:
    nb = nbformat.read(f, as_version=4)

# Check what's in the metadata
print("=== TOP LEVEL METADATA KEYS ===")
print(list(nb.metadata.keys()))

if 'widgets' in nb.metadata:
    print("\n=== WIDGETS METADATA KEYS ===")
    print(list(nb.metadata['widgets'].keys()))

    for key, val in nb.metadata['widgets'].items():
        print(f"\nWidget key: {key}")
        print(f"Type: {type(val)}")
        if isinstance(val, dict):
            print(f"Sub-keys: {list(val.keys())}")
else:
    print("\nNo widgets metadata found")

print(f"\n=== NBFORMAT VERSION ===")
print(f"{nb.nbformat}.{nb.nbformat_minor}")

=== TOP LEVEL METADATA KEYS ===
['colab', 'kernelspec', 'language_info', 'widgets']

=== WIDGETS METADATA KEYS ===
['application/vnd.jupyter.widget-state+json']

Widget key: application/vnd.jupyter.widget-state+json
Type: <class 'nbformat.notebooknode.NotebookNode'>
Sub-keys: ['08301e8b75f84075a949324b5a7ef221', '256560743de5440f845c5d2d09ef3e6b', 'e69e3c1ba786489b97a6aa94df46163f', 'state']

=== NBFORMAT VERSION ===
4.0
